In [ ]:
# Auteur: Habiba Chakour
# Dernière modification: 13 Janvier 2025
# Programme de prédiction des jetons masqués de tous les prompts crées
# Le modele de langue choisi : le génerateur d'Electra-Large

In [ ]:
# Monter google drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#data_path = '/content/drive/MyDrive/GeneratedPrompts/'
data_path = '/content/drive/MyDrive/ToDo/'

predictMlm_path = '/content/drive/MyDrive/scoresElectraGenerator/'
#filesFiltred_path = '/content/drive/MyDrive/filtredFiles/'

In [ ]:
import torch
import os
from pathlib import Path
import pandas as pd
import numpy as np
import array as ar
from transformers import pipeline
import json
from scipy import stats

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cuda')
print(device)

True
1
cuda


In [ ]:
fill_mask = pipeline(
    "fill-mask",
    model="google/electra-large-generator",
    device= device,
    tokenizer="google/electra-large-generator",
    #model="google/electra-base-generator",
    #tokenizer="google/electra-base-generator",
    #model="google/electra-small-generator",
    #tokenizer="google/electra-small-generator",
    top_k=3
)
modelName = fill_mask.model.name_or_path
print(fill_mask.device)

if modelName == "google/electra-small-generator":
     electraModelName = "SmallGeneratorElectra_"
elif modelName == "google/electra-base-generator":
     electraModelName = "BaseGeneratorElectra_"
elif modelName == "google/electra-large-generator":
     electraModelName = "LargeGeneratorElectra_"

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/205M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device set to use cuda


cuda


In [ ]:
# Définition d'une fonction qui retourne les préditions du MLM choisi (ElctraGenerator: Small / Base / Large) dans un objet Dataframe

def promptsFillMask(filePath):

    print(file_path.name)
    promptsFillMask = pd.read_csv(filePath,sep=";", skipinitialspace=True)

    # Extraire les 3 premiers masques avec leurs scores et sequences dans un DataFrame

    aPrompts = promptsFillMask["prompts"]
    aMasks1 = []
    aMasks2= []
    aMasks3 = []

    aScore1 = ar.array('f')
    aScore2 = ar.array('f')
    aScore3 = ar.array('f')

    aSequence1 = []
    aSequence2 = []
    aSequence3 = []

    # Parcourir les prompts du fichier csv
    for i in range(aPrompts.size):
        prompt = aPrompts[i]
        fillMasque = fill_mask(prompt)
        #jsonMasque = json.dumps(fillMasque, indent=4)

        aMasks1.append(fillMasque[0]["token_str"])
        aScore1.append(fillMasque[0]["score"])
        aSequence1.append(fillMasque[0]["sequence"])

        aMasks2.append(fillMasque[1]["token_str"])
        aScore2.append(fillMasque[1]["score"])
        aSequence2.append(fillMasque[1]["sequence"])

        aMasks3.append(fillMasque[2]["token_str"])
        aScore3.append(fillMasque[2]["score"])
        aSequence3.append(fillMasque[2]["sequence"])

    # On récupere les 3 premiers tokens prédits, leurs scores et leurs sequences correspondants

    promptsFillMask['mask1'] = pd.Series(aMasks1)
    promptsFillMask['score1'] = pd.Series(aScore1)
    promptsFillMask['sequence1'] = pd.Series(aSequence1)

    promptsFillMask['mask2'] = pd.Series(aMasks2)
    promptsFillMask['score2'] = pd.Series(aScore2)
    promptsFillMask['sequence2'] = pd.Series(aSequence2)

    promptsFillMask['mask3'] = pd.Series(aMasks3)
    promptsFillMask['score3'] = pd.Series(aScore3)
    promptsFillMask['sequence3'] = pd.Series(aSequence3)

    return promptsFillMask

In [ ]:
# Prédiction des tokens masqués de tous les prompts (les fichiers -.csv) et enregistrement des résultats du MLM dans des fichiers -.csv et -.xslx

# Parcourir la liste des fichiers -.csv (les prompts)
file_paths = list(Path(data_path).glob("*.csv"))
dfPrompts = []
for file_path in file_paths:

  dfPrompts = promptsFillMask(file_path)
  print(dfPrompts.head)

  # extraire le nom du fichier sans l'extention
  fileName,ext = file_path.name.split(sep=".")

  # concatener (le nom du modele Electra choisi + l'extention -.csv ou -.excel)
  outputFileNameCsv = "scoresOf"+electraModelName+fileName+".csv"
  #outputFileNameExcel = "scoresOf"+electraModelName+fileName+".xlsx"

  # Transfert des résultats dans un fichier -.csv/-.excel.
  dfPrompts.to_csv(predictMlm_path + outputFileNameCsv, sep=";", index=False)
  #dfPrompts.to_excel(predictMlm_path + outputFileNameExcel, index=False)


prompts_MS_GroupSpaceHolder_HSconnectors.csv


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


<bound method NDFrame.head of                                            spaceHolder groupe sousGroupe  \
0                             abnormal hearing persons     MS         MS   
1                             abnormal hearing persons     MS         MS   
2                             abnormal hearing persons     MS         MS   
3                             abnormal hearing persons     MS         MS   
4                             abnormal hearing persons     MS         MS   
..                                                 ...    ...        ...   
919  deaf or hard of hearing with additional needs ...     MS         MS   
920  deaf or hard of hearing with additional needs ...     MS         MS   
921  deaf or hard of hearing with additional needs ...     MS         MS   
922  deaf or hard of hearing with additional needs ...     MS         MS   
923  deaf or hard of hearing with additional needs ...     MS         MS   

    ignore categorie  context cat_gr    connector   refer

In [ ]:
# Filtrer les prédictions du -MlmElectra selon la rubriqye -Ignore == non

#def filterPredictMlm(fileToFilter):
    # Lecture du fichier -.csv a filter (les prédictions)
    #dfPredict = pd.read_csv(fileToFilter,sep=";", skipinitialspace=True)
    #dfFiltredPredict = dfPredict.loc[dfPredict['ignore'] == 'non']

    #return dfFiltredPredict

# Parcourir la liste des fichiers -.csv (contenant les predictions)
#filesToFilter_paths = list(Path(predictMlm_path).glob("*.csv"))
#dfFiltred = []

#for fileToFilter_path in filesToFilter_paths:

   #dfFiltred = filterPredictMlm(fileToFilter_path)
   #print(dfFiltred)

   # extraire le nom du fichier sans l'extention
   #fileName,ext = fileToFilter_path.name.split(sep=".")

   # concatener (le nom du modele Electra choisi + l'extention -.csv ou -.excel)
   #outputFileNameCsv = "scoresOf"+electraModelName+fileName+".csv"
   #outputFileNameExcel = "scoresOf"+electraModelName+fileName+".xlsx"

   # Transfert des résultats dans un fichier -.csv/-.excel.
   #dfFiltred.to_csv(filesFiltred_path + outputFileNameCsv, sep=";", index=False)
   #dfFiltred.to_excel(filesFiltred_path + outputFileNameExcel, index=False)

In [ ]:
# Analyse et interprétation des résultats en utilisant le "test-t de student".

# Choisir le modele TweetNLP ou un autre
modele = "TweetNLP"
scores_path = scoresTweetNLP_path
#modele = ""
#scores_path =

# lecture des fichiers scores .csv
file_paths_scores = list(Path(scores_path).glob("*.csv"))

dfScores = []
for file_score in file_paths_scores:
    # extraire le nom du fichier sans l'extention
    fileName,ext = file_score.name.split(sep=".")
    dfScores = pd.read_csv(file_score,sep=";", skipinitialspace=True)

    print(fileName)

    # extraire les données a partir des fichiers résultats .csv
    scoresDG  = dfScores['scoreV1']
    scoresNDG = dfScores['scoreV3']
    #scoresSG  = dfScores['scoreV3']

    # calcul de la staistique t-test:
    t_stat, p_value = stats.ttest_ind(scoresDG, scoresNDG)

    # Interpret the results:
    alpha = 0.05

    print(p_value)

    if p_value < alpha:
           print("Rejetez l’hypothèse nulle ; il existe une différence significative entre les deux groupes.")
    else:
           print("Ne pas rejeter l’hypothèse nulle ; il n'y a pas de différence significative entre les deux groupes.")

